In [1]:
import pathlib

In [2]:
import numpy as np
import pandas as pd

/tmp/ipykernel_2105400/1662815981.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
data_folder = pathlib.Path("../data/processed/biolog/")
ch_pheno_file = data_folder / "phenotypes/ch_phenotypes.tsv"
ch_unitrembl_file = data_folder / "features/uniprot_trembl/uniprot_trembl_ch_features.tsv"
ch_pheno_file.is_file(), ch_unitrembl_file.is_file()

(True, True)

In [4]:
from trait_prediction.main import PhenotypeSet

In [5]:
# Load the phenotype data
phenotypeset = PhenotypeSet.read_data(ch_pheno_file)

In [6]:
phenotype_sizes = []
for phenotype in phenotypeset.phenotypes:
    phenotype_sizes.append(phenotype.phenotype_data.size)
min(phenotype_sizes), max(phenotype_sizes)

(103, 354)

In [7]:
phenotype1 = list(phenotypeset.phenotypes)[0]
phenotype1

Phenotype (name=Carbon-D-Trehalose, category=ch_biolog, size=323)

In [8]:
from trait_prediction.utils import read_generic_features
from trait_prediction.feature_selection import remove_features_with_low_variance, remove_features_with_high_correlation, feature_selection_kbest

In [9]:
VARIANCE_THRESHOLD = 0.01
CORRELATION_THRESHOLD = 0.95
TEST_SIZE = 0.3
N_SPLITS = 5
PHENOTYPE_SAMPLE_SIZE_THRESHOLD = 10
MINOR_CLASS_SAMPLE_SIZE_THRESHOLD = 5
SHAP_MAX_DISPLAY = 10

In [10]:
raw_features = read_generic_features(ch_unitrembl_file)
raw_features, _ = remove_features_with_low_variance(raw_features, VARIANCE_THRESHOLD)

In [11]:
%%time
features = read_generic_features(ch_unitrembl_file)

CPU times: user 1min 50s, sys: 1.85 s, total: 1min 52s
Wall time: 1min 52s


In [12]:
%%time
features, low_var_features1 = remove_features_with_low_variance(features, threshold=VARIANCE_THRESHOLD)

CPU times: user 597 ms, sys: 1.72 s, total: 2.32 s
Wall time: 2.38 s


In [13]:
len(low_var_features1)

129163

In [14]:
%%time
features, correlated_features_dict1 = remove_features_with_high_correlation(features, threshold=CORRELATION_THRESHOLD)

CPU times: user 14min 37s, sys: 2min 5s, total: 16min 43s
Wall time: 16min 56s


In [15]:
features.shape

(341, 3322)

In [16]:
phenotype1.set_feature_data(features, feature_type="generic")

In [17]:
%%time
(
    low_var_features2,
    correlated_features_dict2,
    low_score_features,
) = phenotype1.filter_feature_data(
    variance_threshold=VARIANCE_THRESHOLD,
    correlation_treshold=CORRELATION_THRESHOLD,
    score_func="chi2",
    n_features=1000,
)

CPU times: user 12.4 s, sys: 789 ms, total: 13.1 s
Wall time: 12.6 s


In [18]:
%%time
low_var_features = set(low_var_features1 + low_var_features2)
correlated_features_dict = {
    **correlated_features_dict1,
    **correlated_features_dict2,
}

CPU times: user 17.9 ms, sys: 0 ns, total: 17.9 ms
Wall time: 18 ms


## Benchmarking correlation function

In [19]:
%%time
corr_matrix = raw_features.corr().abs()

CPU times: user 13min 8s, sys: 55.4 s, total: 14min 4s
Wall time: 14min 6s


In [20]:
corr_matrix

,tr|A0A9E8GM07|A0A9E8GM07_9PSED,tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,tr|A0A2N8GX02|A0A2N8GX02_9PSED,tr|A0A9E8GT22|A0A9E8GT22_9PSED,tr|A0A9E8K6N1|A0A9E8K6N1_9PSED,tr|A0A9E8GRF5|A0A9E8GRF5_9PSED,tr|A0A9E8BB96|A0A9E8BB96_9PSED,tr|A0A9E8GRQ8|A0A9E8GRQ8_9PSED,tr|A0A9E8BK21|A0A9E8BK21_9PSED,...,tr|A0A0H3NRM4|A0A0H3NRM4_YERE1,tr|F4MUQ6|F4MUQ6_YEREN,tr|A0A8B6L0C5|A0A8B6L0C5_YEREN,tr|F4N0G1|F4N0G1_YEREN,tr|A0A0H5G3Z2|A0A0H5G3Z2_YEREN,tr|F4MWV5|F4MWV5_YEREN,tr|F4N7V8|F4N7V8_YEREN,tr|F4MXP5|F4MXP5_YEREN,tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,tr|A0A0E8LY24|A0A0E8LY24_YEREN
tr|A0A9E8GM07|A0A9E8GM07_9PSED,1.000000,0.797053,0.869126,0.758812,0.676661,0.718787,0.970324,0.584253,0.968649,0.475623,...,0.033161,0.039814,0.030655,0.027943,0.027943,0.035504,0.033161,0.033161,0.024956,0.043747
tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,0.797053,1.000000,0.830400,0.755270,0.519984,0.694724,0.773400,0.606775,0.744363,0.596727,...,0.026431,0.031734,0.024434,0.022272,0.022272,0.028298,0.026431,0.026431,0.019891,0.034868
tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,0.869126,0.830400,1.000000,0.782307,0.677377,0.827023,0.843334,0.555752,0.824839,0.547243,...,0.028821,0.034604,0.026643,0.024286,0.024286,0.030857,0.028821,0.028821,0.021690,0.038021
tr|A0A2N8GX02|A0A2N8GX02_9PSED,0.758812,0.755270,0.782307,1.000000,0.662068,0.621963,0.736294,0.637755,0.701178,0.465366,...,0.025163,0.030211,0.023262,0.021203,0.021203,0.026941,0.025163,0.025163,0.018937,0.033195
tr|A0A9E8GT22|A0A9E8GT22_9PSED,0.676661,0.519984,0.677377,0.662068,1.000000,0.457936,0.656580,0.716073,0.698561,0.343006,...,0.022439,0.026941,0.020743,0.018908,0.018908,0.024024,0.022439,0.022439,0.016886,0.029602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr|F4MWV5|F4MWV5_YEREN,0.035504,0.028298,0.030857,0.026941,0.024024,0.025520,0.036590,0.020743,0.034391,0.016886,...,0.934013,0.891737,0.421347,0.787032,0.787032,1.000000,0.934013,0.934013,0.702898,0.811578
tr|F4N7V8|F4N7V8_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|F4MXP5|F4MXP5_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,0.024956,0.019891,0.021690,0.018937,0.016886,0.017938,0.025719,0.014580,0.024173,0.011869,...,0.752557,0.626800,0.399745,0.666502,0.666502,0.702898,0.752557,0.752557,1.000000,0.570456


In [21]:
del corr_matrix

In [22]:
def pearson_correlation_coefficient(X):
    """
    calculate pearson correlation coefficient of matrix X
    X: numpy array (MxN)
    return pcc: numpy array (MxM)
    """
    M, N = X.shape[0], X.shape[1]  # number of features, number of data points
    X_mean = np.mean(X, axis=1).reshape(M, 1)
    X_std = np.std(X, axis=1).reshape(M, 1)
    X_tilde = (X-X_mean)/X_std
    pcc = X_tilde@X_tilde.T/N
    np.fill_diagonal(pcc, 1, wrap=False)
    return pcc

In [23]:
%%time
corr_matrix = pd.DataFrame(np.abs(pearson_correlation_coefficient(raw_features.values.T)), index=raw_features.columns, columns=raw_features.columns)

CPU times: user 54.1 s, sys: 2min 12s, total: 3min 6s
Wall time: 36.2 s


In [24]:
corr_matrix

,tr|A0A9E8GM07|A0A9E8GM07_9PSED,tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,tr|A0A2N8GX02|A0A2N8GX02_9PSED,tr|A0A9E8GT22|A0A9E8GT22_9PSED,tr|A0A9E8K6N1|A0A9E8K6N1_9PSED,tr|A0A9E8GRF5|A0A9E8GRF5_9PSED,tr|A0A9E8BB96|A0A9E8BB96_9PSED,tr|A0A9E8GRQ8|A0A9E8GRQ8_9PSED,tr|A0A9E8BK21|A0A9E8BK21_9PSED,...,tr|A0A0H3NRM4|A0A0H3NRM4_YERE1,tr|F4MUQ6|F4MUQ6_YEREN,tr|A0A8B6L0C5|A0A8B6L0C5_YEREN,tr|F4N0G1|F4N0G1_YEREN,tr|A0A0H5G3Z2|A0A0H5G3Z2_YEREN,tr|F4MWV5|F4MWV5_YEREN,tr|F4N7V8|F4N7V8_YEREN,tr|F4MXP5|F4MXP5_YEREN,tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,tr|A0A0E8LY24|A0A0E8LY24_YEREN
tr|A0A9E8GM07|A0A9E8GM07_9PSED,1.000000,0.797053,0.869126,0.758812,0.676661,0.718787,0.970324,0.584253,0.968649,0.475623,...,0.033161,0.039814,0.030655,0.027943,0.027943,0.035504,0.033161,0.033161,0.024956,0.043747
tr|A0A9E8H6A6|A0A9E8H6A6_9PSED,0.797053,1.000000,0.830400,0.755270,0.519984,0.694724,0.773400,0.606775,0.744363,0.596727,...,0.026431,0.031734,0.024434,0.022272,0.022272,0.028298,0.026431,0.026431,0.019891,0.034868
tr|A0A9E8GZH5|A0A9E8GZH5_9PSED,0.869126,0.830400,1.000000,0.782307,0.677377,0.827023,0.843334,0.555752,0.824839,0.547243,...,0.028821,0.034604,0.026643,0.024286,0.024286,0.030857,0.028821,0.028821,0.021690,0.038021
tr|A0A2N8GX02|A0A2N8GX02_9PSED,0.758812,0.755270,0.782307,1.000000,0.662068,0.621963,0.736294,0.637755,0.701178,0.465366,...,0.025163,0.030211,0.023262,0.021203,0.021203,0.026941,0.025163,0.025163,0.018937,0.033195
tr|A0A9E8GT22|A0A9E8GT22_9PSED,0.676661,0.519984,0.677377,0.662068,1.000000,0.457936,0.656580,0.716073,0.698561,0.343006,...,0.022439,0.026941,0.020743,0.018908,0.018908,0.024024,0.022439,0.022439,0.016886,0.029602
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
tr|F4MWV5|F4MWV5_YEREN,0.035504,0.028298,0.030857,0.026941,0.024024,0.025520,0.036590,0.020743,0.034391,0.016886,...,0.934013,0.891737,0.421347,0.787032,0.787032,1.000000,0.934013,0.934013,0.702898,0.811578
tr|F4N7V8|F4N7V8_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|F4MXP5|F4MXP5_YEREN,0.033161,0.026431,0.028821,0.025163,0.022439,0.023836,0.034175,0.019374,0.032121,0.015772,...,1.000000,0.832894,0.295229,0.842635,0.842635,0.934013,1.000000,1.000000,0.752557,0.758024
tr|A0A7U0ATN0|A0A7U0ATN0_YEREN,0.024956,0.019891,0.021690,0.018937,0.016886,0.017938,0.025719,0.014580,0.024173,0.011869,...,0.752557,0.626800,0.399745,0.666502,0.666502,0.702898,0.752557,0.752557,1.000000,0.570456


In [25]:
%%time
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
corr_group_dict = {}
cols_to_drop_set = set()
for col in upper.columns:
    corr_filter = upper[col] > CORRELATION_THRESHOLD
    correlated_cols = list(upper.columns[corr_filter])
    if len(correlated_cols) > 0:
        corr_group_dict[col] = correlated_cols
        cols_to_drop_set.add(col)

: 